<a href="https://colab.research.google.com/github/sakuna47/RDB_DA/blob/DV_Code/RDB_DA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import seaborn as sns
from matplotlib.dates import MonthLocator, DateFormatter
import matplotlib.ticker as ticker

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set style for better looking plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

def load_and_process_data(file_path):
    """
    Load and process the CSV file to extract all financial rates
    """
    try:
        # Try reading with different encodings and separators
        for encoding in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']:
            try:
                df = pd.read_csv(file_path, encoding=encoding, header=None)
                print(f"Successfully loaded CSV with encoding: {encoding}")
                break
            except UnicodeDecodeError:
                continue
        else:
            df = pd.read_csv(file_path, header=None)
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        for sep in [',', ';', '\t']:
            try:
                df = pd.read_csv(file_path, sep=sep, header=None)
                print(f"Successfully loaded CSV with separator: '{sep}'")
                break
            except:
                continue
        else:
            raise ValueError("Could not read the CSV file with any common format")

    print("Data shape:", df.shape)

     # Find key row indices
    date_row_idx = None
    usd_lkr_buying_idx = None
    usd_lkr_selling_idx = None
    treasury_bill_91_idx = None
    treasury_bill_182_idx = None
    treasury_bill_364_idx = None
    treasury_bond_2yr_idx = None
    treasury_bond_3yr_idx = None
    treasury_bond_4yr_idx = None
    treasury_bond_5yr_idx = None

 # Search for all required rows
    for idx, row in df.iterrows():
        row_str = ' '.join([str(cell) for cell in row if pd.notna(cell)])

        # Date row
        if 'Date' in str(row.iloc[0]) and '24.07.2024' in row_str:
            date_row_idx = idx
            print(f"Found date row at index: {idx}")

        # USD/LKR rates
        if 'USD/LKR Rate' in str(row.iloc[0]) and 'Buying' in str(row.iloc[1]):
            usd_lkr_buying_idx = idx
            print(f"Found USD/LKR Buying row at index: {idx}")
        elif usd_lkr_buying_idx is not None and idx == usd_lkr_buying_idx + 1 and 'Selling' in str(row.iloc[1]):
            usd_lkr_selling_idx = idx
            print(f"Found USD/LKR Selling row at index: {idx}")

        # Treasury Bills - Fixed pattern matching
        if 'Treasury Bills' in str(row.iloc[0]) and '91 Days' in str(row.iloc[1]):
            treasury_bill_91_idx = idx
            print(f"Found Treasury Bills 91 Days row at index: {idx}")
        elif str(row.iloc[1]).strip() == '182 Days' and treasury_bill_91_idx is not None:
            treasury_bill_182_idx = idx
            print(f"Found Treasury Bills 182 Days row at index: {idx}")
        elif str(row.iloc[1]).strip() == '364 Days' and treasury_bill_182_idx is not None:
            treasury_bill_364_idx = idx
            print(f"Found Treasury Bills 364 Days row at index: {idx}")

        # Treasury Bonds - Fixed pattern matching
        if 'Treasury Bonds' in str(row.iloc[0]) and '2 Yrs' in str(row.iloc[1]):
            treasury_bond_2yr_idx = idx
            print(f"Found Treasury Bonds 2 Yrs row at index: {idx}")
        elif str(row.iloc[1]).strip() == '3 Yrs' and treasury_bond_2yr_idx is not None:
            treasury_bond_3yr_idx = idx
            print(f"Found Treasury Bonds 3 Yrs row at index: {idx}")
        elif str(row.iloc[1]).strip() == '4 Yrs' and treasury_bond_3yr_idx is not None:
            treasury_bond_4yr_idx = idx
            print(f"Found Treasury Bonds 4 Yrs row at index: {idx}")
        elif str(row.iloc[1]).strip() == '5 Yrs' and treasury_bond_4yr_idx is not None:
            treasury_bond_5yr_idx = idx
            print(f"Found Treasury Bonds 5 Yrs row at index: {idx}")

    if date_row_idx is None:
        raise ValueError("Could not find date row in the data")

     # Extract date headers
    date_row = df.iloc[date_row_idx]
    start_col = 2  # Start from column 2 based on CSV structure

    dates = []
    for col_idx in range(start_col, len(date_row)):
        date_str = str(date_row.iloc[col_idx])
        if date_str == 'nan' or date_str == '' or date_str == '-':
            continue

        try:
            for date_format in ['%d.%m.%Y', '%d/%m/%Y', '%Y-%m-%d', '%m/%d/%Y']:
                try:
                    parsed_date = datetime.strptime(date_str, date_format)
                    dates.append(parsed_date)
                    break
                except ValueError:
                    continue
        except:
            continue

    print(f"Successfully extracted {len(dates)} dates")

     # Helper function to extract rates for a specific row
    def extract_rates(row_idx, dates):
        if row_idx is None:
            return [None] * len(dates)

        row = df.iloc[row_idx]
        rates = []

        for col_idx in range(start_col, start_col + len(dates)):
            if col_idx >= len(row):
                rates.append(None)
                continue

            rate_str = str(row.iloc[col_idx])
            if rate_str == 'nan' or rate_str == '' or rate_str == '-':
                rates.append(None)
                continue

            try:
                # Handle special cases with multiple values or ranges
                if '/' in rate_str:
                    # Take the first value for ranges like "9.25/9.50"
                    rate_str = rate_str.split('/')[0]

                rate_clean = rate_str.replace(',', '').replace(' ', '')
                rates.append(float(rate_clean))
            except ValueError:
                rates.append(None)

        return rate


    # Extract all rates
    data = {
        'dates': dates,
        'usd_lkr_buying': extract_rates(usd_lkr_buying_idx, dates),
        'usd_lkr_selling': extract_rates(usd_lkr_selling_idx, dates),
        'treasury_bill_91': extract_rates(treasury_bill_91_idx, dates),
        'treasury_bill_182': extract_rates(treasury_bill_182_idx, dates),
        'treasury_bill_364': extract_rates(treasury_bill_364_idx, dates),
        'treasury_bond_2yr': extract_rates(treasury_bond_2yr_idx, dates),
        'treasury_bond_3yr': extract_rates(treasury_bond_3yr_idx, dates),
        'treasury_bond_4yr': extract_rates(treasury_bond_4yr_idx, dates),
        'treasury_bond_5yr': extract_rates(treasury_bond_5yr_idx, dates)
    }

    # Print summary of found data
    print("\nData Summary:")
    for key, values in data.items():
        if key != 'dates':
            valid_count = len([x for x in values if x is not None])
            print(f"  {key}: {valid_count} valid data points")

    return data
def setup_detailed_axes(ax, data_type='exchange_rate'):
    """
    Setup detailed X and Y axes with more months and decimal precision
    """
    # X-axis: Show more months with detailed formatting
    ax.xaxis.set_major_locator(MonthLocator(interval=1))  # Show every month
    ax.xaxis.set_minor_locator(MonthLocator(interval=1))  # Minor ticks every month
    ax.xaxis.set_major_formatter(DateFormatter('%b %Y'))  # Format: Jan 2024

    # Rotate x-axis labels for better readability
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Y-axis: Show more decimal points based on data type
    if data_type == 'exchange_rate':
        # For exchange rates, show 2 decimal places
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(5))
    else:
        # For interest rates, show 2 decimal places
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f%%'))
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(5))

    # Add minor grid lines for better precision reading
    ax.grid(True, which='major', linestyle='-', alpha=0.3)
    ax.grid(True, which='minor', linestyle=':', alpha=0.2)

def create_usd_lkr_chart(data):
    """Create USD/LKR exchange rate chart with detailed axes"""
    fig, ax = plt.subplots(figsize=(16, 10))

    dates = data['dates']
    buying_rates = [x for x in data['usd_lkr_buying'] if x is not None]
    selling_rates = [x for x in data['usd_lkr_selling'] if x is not None]
    valid_dates_buying = [dates[i] for i in range(len(dates)) if data['usd_lkr_buying'][i] is not None]
    valid_dates_selling = [dates[i] for i in range(len(dates)) if data['usd_lkr_selling'][i] is not None]

    ax.plot(valid_dates_buying, buying_rates, marker='o', linewidth=2.5, markersize=5,
            label='USD/LKR Buying', color='#2E86AB', alpha=0.8)
    ax.plot(valid_dates_selling, selling_rates, marker='s', linewidth=2.5, markersize=5,
            label='USD/LKR Selling', color='#F24236', alpha=0.8)

    ax.set_title('USD/LKR Exchange Rate Over Time', fontsize=18, fontweight='bold', pad=25)
    ax.set_xlabel('Date', fontsize=14, fontweight='bold')
    ax.set_ylabel('USD/LKR Rate', fontsize=14, fontweight='bold')